In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas_datareader.data as web
import datetime

from functions import get_polymarket_event

In [2]:
june_event, june_probs = get_polymarket_event("fed-decision-in-june-825", "2026-06-17")
for name, df in june_probs.items():
    print(name, len(df), "points")

50+ bps decrease 240 points
25 bps decrease 240 points
No change 240 points
25 bps increase 240 points
50+ bps increase 240 points


In [3]:
start_june = datetime.datetime(2026, 5, 17)
end_june = datetime.datetime(2026, 6, 16)

effr_june = web.DataReader('EFFR', 'fred', start_june, end_june)
r_before_june = effr_june['EFFR'].mean()
print(r_before_june)

3.6214285714285714


In [4]:
combined_june = pd.DataFrame()
for outcome_name, odf in june_probs.items():
    s = odf.set_index('datetime')['p'].resample('1min').last().ffill()
    combined_june[outcome_name] = s

combined_june = combined_june.dropna()

prob_sum_june = (
    combined_june['50+ bps decrease'] +
    combined_june['25 bps decrease'] +
    combined_june['No change'] +
    combined_june['25 bps increase'] +
    combined_june['50+ bps increase']
)

combined_june['r_after'] = (
    (combined_june['50+ bps decrease'] / prob_sum_june) * (r_before_june - 0.50) +
    (combined_june['25 bps decrease'] / prob_sum_june) * (r_before_june - 0.25) +
    (combined_june['No change'] / prob_sum_june) * r_before_june +
    (combined_june['25 bps increase'] / prob_sum_june) * (r_before_june + 0.25) +
    (combined_june['50+ bps increase'] / prob_sum_june) * (r_before_june + 0.50)
)

print(combined_june['r_after'].describe())

count    240.000000
mean       3.621238
std        0.000230
min        3.620929
25%        3.620929
50%        3.621429
75%        3.621429
max        3.621553
Name: r_after, dtype: float64


In [5]:
april_event, april_probs = get_polymarket_event("fed-decision-in-april", "2026-04-29")
for name, df in april_probs.items():
    print(name, len(df), "points")

50+ bps decrease 240 points
25 bps decrease 240 points
No change 240 points
25+ bps increase 240 points


In [6]:
start_april = datetime.datetime(2026, 3, 29)
end_april = datetime.datetime(2026, 4, 28)

effr_april = web.DataReader('EFFR', 'fred', start_april, end_april)
r_before_april = effr_april['EFFR'].mean()
print(r_before_april)

3.64


In [7]:
combined_april = pd.DataFrame()
for outcome_name, odf in april_probs.items():
    s = odf.set_index('datetime')['p'].resample('1min').last().ffill()
    combined_april[outcome_name] = s

combined_april = combined_april.dropna()

prob_sum_april = (
    combined_april['50+ bps decrease'] +
    combined_april['25 bps decrease'] +
    combined_april['No change'] +
    combined_april['25+ bps increase']
)

combined_april['r_after'] = (
    (combined_april['50+ bps decrease'] / prob_sum_april) * (r_before_april - 0.50) +
    (combined_april['25 bps decrease'] / prob_sum_april) * (r_before_april - 0.25) +
    (combined_april['No change'] / prob_sum_april) * r_before_april +
    (combined_april['25+ bps increase'] / prob_sum_april) * (r_before_april + 0.25)
)

print(combined_april['r_after'].describe())

count    240.000000
mean       3.639757
std        0.000195
min        3.639251
25%        3.639750
50%        3.639750
75%        3.639875
max        3.640623
Name: r_after, dtype: float64


In [8]:
june_range = combined_june['r_after'].max() - combined_june['r_after'].min()
april_range = combined_april['r_after'].max() - combined_april['r_after'].min()

print("july 2026:  ~0.08 percentage points")
print(f"march 2026: ~0.0014 percentage points")
print(f"june 2026:  {june_range:.4f} percentage points")
print(f"april 2026: {april_range:.4f} percentage points")

july 2026:  ~0.08 percentage points
march 2026: ~0.0014 percentage points
june 2026:  0.0006 percentage points
april 2026: 0.0014 percentage points
